In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# Settings
# ============================================================

summary_path = Path(
    "results/summary/exid_metrics_summary_all_recordings.csv"
)

# From drone-dataset-tools-master/src
data_dir = Path("data")

# Output files
diagnostic_path = Path(
    "results/summary/data_cleaning_diagnostics.csv"
)

flagged_path = Path(
    "results/summary/excluded_vehicles.csv"
)

cleaned_path = Path(
    "results/summary/exid_metrics_summary_all_recordings_cleaned.csv"
)

# ----------------------------
# Heading-consistency settings
# ----------------------------

# Ignore almost stationary frames because velocity direction becomes unstable
MIN_SPEED = 2.0  # m/s

# Persistent ~180-degree discrepancy
FLIP_ANGLE_THRESHOLD = 150.0  # degrees
FLIP_FRAME_FRACTION = 0.80    # at least 80% of checked frames

# ----------------------------
# Duration / censoring settings
# ----------------------------

# Short duration itself does NOT cause removal.
# It is only a flag when combined with the recording-boundary condition.
SHORT_DURATION_THRESHOLD = 5.0  # seconds

# Allow a small tolerance around the final frame of the recording
END_FRAME_TOLERANCE = 2  # frames


# ============================================================
# Load summary
# ============================================================

summary = pd.read_csv(summary_path)

print("==========================================")
print("Original dataset")
print("==========================================")
print(f"Vehicles: {len(summary):,}")
print(f"Recordings: {summary['recording_id'].nunique()}")

print(
    "Vehicles with duration < 5 s:",
    int((summary["duration_s"] < SHORT_DURATION_THRESHOLD).sum())
)

print(
    "Vehicles with negative average speed:",
    int((summary["average_speed"] < 0).sum())
)

print(
    "Vehicles with negative maximum speed:",
    int((summary["max_speed"] < 0).sum())
)

In [ ]:
# ============================================================
# Check every merging vehicle
# ============================================================

diagnostics = []

for recording_id, recording_summary in summary.groupby("recording_id"):

    recording_id = int(recording_id)

    tracks_path = data_dir / f"{recording_id:02d}_tracks.csv"

    if not tracks_path.exists():
        print(
            f"Warning: tracks file not found for recording "
            f"{recording_id:02d}: {tracks_path}"
        )
        continue

    tracks = pd.read_csv(tracks_path)

    # Actual first and last available frames in this recording
    recording_first_frame = int(tracks["frame"].min())
    recording_last_frame = int(tracks["frame"].max())

    print(
        f"Checking recording {recording_id:02d} | "
        f"{len(recording_summary)} merging vehicles | "
        f"frames {recording_first_frame}-{recording_last_frame}"
    )

    # Only retain tracks corresponding to vehicles in the merging summary
    merging_ids = set(
        recording_summary["ego_track_id"].astype(int)
    )

    merging_tracks = tracks[
        tracks["trackId"].isin(merging_ids)
    ].copy()

    for _, vehicle_summary in recording_summary.iterrows():

        track_id = int(vehicle_summary["ego_track_id"])

        frame_start = int(vehicle_summary["frame_start"])
        frame_end = int(vehicle_summary["frame_end"])
        duration_s = float(vehicle_summary["duration_s"])

        vehicle = merging_tracks[
            (merging_tracks["trackId"] == track_id)
            & (merging_tracks["frame"] >= frame_start)
            & (merging_tracks["frame"] <= frame_end)
        ].copy()

        # ====================================================
        # 1. Duration / right-censoring check
        # ====================================================

        is_short_duration = (
            duration_s < SHORT_DURATION_THRESHOLD
        )

        frames_to_recording_end = (
            recording_last_frame - frame_end
        )

        ends_at_recording_boundary = (
            frames_to_recording_end <= END_FRAME_TOLERANCE
        )

        # Only classify it as likely censored when BOTH conditions hold
        likely_right_censored = (
            is_short_duration
            and ends_at_recording_boundary
        )

        # ====================================================
        # 2. Heading-consistency check
        # ====================================================

        orientation_status = "ok"

        num_checked_frames = 0
        median_abs_heading_difference = np.nan
        flip_fraction = np.nan
        mean_speed_magnitude = np.nan
        mean_lon_velocity = np.nan

        if vehicle.empty:

            orientation_status = "no trajectory data"

        else:

            speed_magnitude = np.sqrt(
                vehicle["xVelocity"] ** 2
                + vehicle["yVelocity"] ** 2
            )

            valid = (
                (speed_magnitude >= MIN_SPEED)
                & vehicle[
                    [
                        "heading",
                        "xVelocity",
                        "yVelocity"
                    ]
                ].notna().all(axis=1)
            )

            check = vehicle.loc[valid].copy()
            check_speed = speed_magnitude.loc[valid]

            if check.empty:

                orientation_status = "insufficient moving frames"

            else:

                num_checked_frames = len(check)

                # --------------------------------------------
                # Actual motion direction
                # --------------------------------------------

                motion_heading = (
                    np.degrees(
                        np.arctan2(
                            check["yVelocity"],
                            check["xVelocity"]
                        )
                    ) % 360
                )

                # --------------------------------------------
                # Circular difference:
                # result lies between -180 and +180 deg
                # --------------------------------------------

                heading_difference = (
                    (
                        motion_heading
                        - check["heading"]
                        + 180
                    ) % 360
                ) - 180

                abs_difference = np.abs(
                    heading_difference
                )

                median_abs_heading_difference = float(
                    np.median(abs_difference)
                )

                flip_fraction = float(
                    np.mean(
                        abs_difference
                        >= FLIP_ANGLE_THRESHOLD
                    )
                )

                mean_speed_magnitude = float(
                    check_speed.mean()
                )

                if "lonVelocity" in check.columns:
                    mean_lon_velocity = float(
                        check["lonVelocity"].mean()
                    )

                if (
                    median_abs_heading_difference
                    >= FLIP_ANGLE_THRESHOLD
                    and
                    flip_fraction
                    >= FLIP_FRAME_FRACTION
                ):
                    orientation_status = "heading_flip"

        # ====================================================
        # 3. Overall cleaning decision
        # ====================================================

        heading_problem = (
            orientation_status == "heading_flip"
        )

        exclude_vehicle = (
            heading_problem
            or likely_right_censored
        )

        exclusion_reasons = []

        if heading_problem:
            exclusion_reasons.append(
                "heading_orientation_inconsistency"
            )

        if likely_right_censored:
            exclusion_reasons.append(
                "short_duration_right_censored"
            )

        if exclusion_reasons:
            exclusion_reason = "; ".join(
                exclusion_reasons
            )
        else:
            exclusion_reason = ""

        # ====================================================
        # Save diagnostics
        # ====================================================

        diagnostics.append({

            "recording_id":
                recording_id,

            "ego_track_id":
                track_id,

            "duration_s":
                duration_s,

            "frame_start":
                frame_start,

            "frame_end":
                frame_end,

            "recording_first_frame":
                recording_first_frame,

            "recording_last_frame":
                recording_last_frame,

            "frames_to_recording_end":
                frames_to_recording_end,

            "is_short_duration":
                is_short_duration,

            "ends_at_recording_boundary":
                ends_at_recording_boundary,

            "likely_right_censored":
                likely_right_censored,

            "num_checked_frames":
                num_checked_frames,

            "median_abs_heading_difference_deg":
                median_abs_heading_difference,

            "flip_fraction":
                flip_fraction,

            "mean_speed_magnitude":
                mean_speed_magnitude,

            "mean_lonVelocity":
                mean_lon_velocity,

            "summary_average_speed":
                vehicle_summary["average_speed"],

            "summary_max_speed":
                vehicle_summary["max_speed"],

            "orientation_status":
                orientation_status,

            "exclude_vehicle":
                exclude_vehicle,

            "exclusion_reason":
                exclusion_reason
        })

In [ ]:
# ============================================================
# Create diagnostic table
# ============================================================

diagnostics_df = pd.DataFrame(diagnostics)

diagnostics_df.to_csv(
    diagnostic_path,
    index=False
)

print("\n==========================================")
print("Data-cleaning diagnostics")
print("==========================================")

print(
    "Vehicles checked:",
    len(diagnostics_df)
)

print(
    "Short-duration vehicles:",
    int(
        diagnostics_df[
            "is_short_duration"
        ].sum()
    )
)

print(
    "Short + recording-end censored:",
    int(
        diagnostics_df[
            "likely_right_censored"
        ].sum()
    )
)

print(
    "Heading-flipped vehicles:",
    int(
        (
            diagnostics_df[
                "orientation_status"
            ] == "heading_flip"
        ).sum()
    )
)

print(
    "Total vehicles marked for exclusion:",
    int(
        diagnostics_df[
            "exclude_vehicle"
        ].sum()
    )
)

In [ ]:
short_cases = diagnostics_df[
    diagnostics_df["is_short_duration"]
].copy()

display(
    short_cases[
        [
            "recording_id",
            "ego_track_id",
            "duration_s",
            "frame_start",
            "frame_end",
            "recording_last_frame",
            "frames_to_recording_end",
            "likely_right_censored"
        ]
    ].sort_values(
        [
            "duration_s",
            "recording_id"
        ]
    )
)

In [ ]:
flagged = diagnostics_df[
    diagnostics_df["exclude_vehicle"]
].copy()

flagged.to_csv(
    flagged_path,
    index=False
)

display(
    flagged[
        [
            "recording_id",
            "ego_track_id",
            "duration_s",
            "median_abs_heading_difference_deg",
            "flip_fraction",
            "frames_to_recording_end",
            "exclusion_reason"
        ]
    ]
)

In [ ]:
# ============================================================
# Remove flagged vehicles
# ============================================================

flagged_keys = flagged[
    [
        "recording_id",
        "ego_track_id"
    ]
].drop_duplicates()

cleaned_summary = summary.merge(
    flagged_keys.assign(
        remove_from_analysis=True
    ),
    on=[
        "recording_id",
        "ego_track_id"
    ],
    how="left"
)

cleaned_summary = cleaned_summary[
    cleaned_summary[
        "remove_from_analysis"
    ].isna()
].drop(
    columns="remove_from_analysis"
).reset_index(
    drop=True
)

cleaned_summary.to_csv(
    cleaned_path,
    index=False
)

print("\n==========================================")
print("Final cleaned dataset")
print("==========================================")

print(
    f"Before cleaning: "
    f"{len(summary):,}"
)

print(
    f"Removed: "
    f"{len(flagged_keys):,}"
)

print(
    f"After cleaning: "
    f"{len(cleaned_summary):,}"
)

print(
    "\nRemoval reasons:"
)

print(
    flagged[
        "exclusion_reason"
    ].value_counts()
)

print(
    "\nRemaining negative average speeds:",
    int(
        (
            cleaned_summary[
                "average_speed"
            ] < 0
        ).sum()
    )
)

print(
    "\nSaved diagnostic file:"
)
print(
    diagnostic_path.resolve()
)

print(
    "\nSaved excluded vehicles:"
)
print(
    flagged_path.resolve()
)

print(
    "\nSaved cleaned summary:"
)
print(
    cleaned_path.resolve()
)